In [0]:
from pyspark.sql import functions as F

# =========================================================
# CONFIGURATION
# =========================================================

SEED = 42
BASE_PATH = "/Volumes/workspace/finance_analytics/finance_raw"

# =========================================================
# LOAD LOAN DATA
# =========================================================

loans_df = spark.read.parquet(
    f"{BASE_PATH}/loans"
)

print(f"Loans loaded: {loans_df.count():,}")

# =========================================================
# GENERATE LOAN PAYMENTS
# =========================================================

loan_payments_df = (
    spark.range(1, 100001)
    .withColumnRenamed("id", "payment_id")

    # Link each payment to an existing loan
    .withColumn(
        "loan_id",
        (F.rand(SEED) * 8000).cast("int") + 1
    )

    # Payment date across 2024-2025
    .withColumn(
        "payment_date",
        F.date_add(
            F.to_date(F.lit("2024-01-01")),
            (F.rand(SEED + 1) * 731).cast("int")
        )
    )

    # Payment amount between €100 and €5,000
    .withColumn(
        "payment_amount",
        F.round(
            F.rand(SEED + 2) * 4900 + 100,
            2
        )
    )

    # Interest portion: approximately 10%-30%
    .withColumn(
        "interest_amount",
        F.round(
            F.col("payment_amount") *
            (F.rand(SEED + 3) * 0.20 + 0.10),
            2
        )
    )

    # Principal = payment - interest
    .withColumn(
        "principal_amount",
        F.round(
            F.col("payment_amount") -
            F.col("interest_amount"),
            2
        )
    )

    # Payment status
    .withColumn(
        "payment_status",
        F.when(F.rand(SEED + 4) < 0.82, "Paid")
         .when(F.rand(SEED + 4) < 0.94, "Late")
         .when(F.rand(SEED + 4) < 0.98, "Missed")
         .otherwise("Partial")
    )

    # Days past due
    .withColumn(
        "days_past_due",
        F.when(
            F.col("payment_status") == "Paid",
            0
        )
        .when(
            F.col("payment_status") == "Late",
            (F.rand(SEED + 5) * 30 + 1).cast("int")
        )
        .when(
            F.col("payment_status") == "Missed",
            (F.rand(SEED + 6) * 90 + 31).cast("int")
        )
        .otherwise(
            (F.rand(SEED + 7) * 15 + 1).cast("int")
        )
    )
)

# =========================================================
# FINAL COLUMN ORDER
# =========================================================

loan_payments_df = loan_payments_df.select(
    "payment_id",
    "loan_id",
    "payment_date",
    "payment_amount",
    "principal_amount",
    "interest_amount",
    "payment_status",
    "days_past_due"
)

display(loan_payments_df.limit(20))

In [0]:
# =========================================================
# LOAN PAYMENT VALIDATION
# =========================================================

print(f"Payment count: {loan_payments_df.count():,}")

print("Payment status distribution:")

display(
    loan_payments_df
    .groupBy("payment_status")
    .count()
    .orderBy("payment_status")
)

# Check loan references
invalid_loans = (
    loan_payments_df
    .join(
        loans_df.select("loan_id"),
        on="loan_id",
        how="left_anti"
    )
)

print(
    f"Invalid loan references: "
    f"{invalid_loans.count()}"
)

# Check payment calculation
calculation_errors = (
    loan_payments_df
    .filter(
        F.abs(
            F.col("payment_amount") -
            F.col("principal_amount") -
            F.col("interest_amount")
        ) > 0.01
    )
)

print(
    f"Payment calculation errors: "
    f"{calculation_errors.count()}"
)

In [0]:
# =========================================================
# SAVE LOAN PAYMENT DATA TO FINANCE RAW VOLUME
# =========================================================

loan_payments_df.write.mode("overwrite").parquet(
    f"{BASE_PATH}/loan_payments"
)

print("Loan payment data successfully written to finance_raw Volume.")

In [0]:
# =========================================================
# VERIFY SAVED LOAN PAYMENT DATA
# =========================================================

saved_payments_df = spark.read.parquet(
    f"{BASE_PATH}/loan_payments"
)

print(f"Saved payments: {saved_payments_df.count():,}")

display(saved_payments_df.limit(10))